In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification
from transformers import ASTModel
from transformers import DefaultDataCollator
#from datasets import load_metric
import evaluate
from transformers import Trainer, TrainingArguments
import os
from transformers import EarlyStoppingCallback

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0511 21:24:13.910000 24548 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
pip install transformers==4.39.3

In [2]:
data_path = Path(r"C:\Users\Kochana\projects\genres\data\gtzan\gtzan.npz")
data = np.load(data_path)
lst = data.files

In [3]:
tracks_path = []
labels = []
#data_path = Path(r"/content/drive/MyDrive/data/gtzan_old")
for path in lst:
    #path_file = os.path.join(data_path, file, "track.npy")
    tracks_path.append(path)
    labels.append(path[6:][:-12])
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
train, validation, train_labels, val_labels = train_test_split(
        tracks_path, encoded_labels, test_size=0.2, stratify=encoded_labels, random_state=42)


In [4]:
import wandb
wandb.init(project="ast_model", name="wadims_pc_astmodel_whole_model")

wandb: Currently logged in as: polinaitsme (polinaitsme-RWTH Aachen University) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
class GTZANSpectrogramDataset(Dataset):
    def __init__(self, path, labels):
        self.paths = path
        self.labels = labels
        self.max_time = 1020
        self.data = data
        
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        #spec = self.spectrograms[idx]  # shape: (128, time)
        spec = data[self.paths[idx]]
        #print(spec.shape)
        #spec = spec[None, :, :]  # add channel dim
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec.squeeze(0)
        spec = spec[:self.max_time, :]
        spec = torch.tensor(spec, dtype=torch.float32)

        #spec = spec.unsqueeze(0)
        label = self.labels[idx]
        return {"input_values": spec, "labels": int(label)}

# model = ASTModel.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593",
#     num_labels=10,  # GTZAN has 10 genres
#     ignore_mismatched_sizes=True  # allows adjusting output layer
# )
data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./ast-gtzan",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=80,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    gradient_accumulation_steps=8,
    greater_is_better=True,
    report_to="wandb",
)
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, val_labels)

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class ASTForGenreClassification(nn.Module):
    def __init__(self, ast_model, num_labels=10):
        super().__init__()
        self.ast = ast_model
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_values, labels=None):
        #with torch.no_grad():
        x = self.ast.embeddings(input_values)  # patchify
        x = self.ast.encoder(x).last_hidden_state  # (B, T, 768)
        x = x.mean(dim=1)  # simple mean pooling (or use CLS)
        logits = self.classifier(x)

        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return loss, logits
        else:
            return logits


In [7]:
base_ast = ASTModel.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    ignore_mismatched_sizes=True
)
model = ASTForGenreClassification(ast_model=base_ast, num_labels=10)

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)
trainer.train()

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\accelerate\accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
                                                  
  1%|▏         | 50/4000 [00:48<56:21,  1.17it/s]

{'eval_loss': 1.1230772733688354, 'eval_accuracy': 0.64, 'eval_runtime': 4.4239, 'eval_samples_per_second': 45.209, 'eval_steps_per_second': 22.605, 'epoch': 1.0}


                                                   
  2%|▎         | 100/4000 [01:37<55:25,  1.17it/s]

{'eval_loss': 0.7513218522071838, 'eval_accuracy': 0.74, 'eval_runtime': 4.4031, 'eval_samples_per_second': 45.422, 'eval_steps_per_second': 22.711, 'epoch': 2.0}


                                                    
  4%|▍         | 150/4000 [02:26<54:55,  1.17it/s]

...

                                                     
 29%|██▉       | 1150/4000 [18:36<40:22,  1.18it/s]

{'eval_loss': 1.138649582862854, 'eval_accuracy': 0.82, 'eval_runtime': 4.4135, 'eval_samples_per_second': 45.316, 'eval_steps_per_second': 22.658, 'epoch': 23.0}


 29%|██▉       | 1150/4000 [18:37<46:09,  1.03it/s]

{'train_runtime': 1117.4422, 'train_samples_per_second': 57.202, 'train_steps_per_second': 3.58, 'train_loss': 0.1896103126069774, 'epoch': 23.0}


TrainOutput(global_step=1150, training_loss=0.1896103126069774, metrics={'train_runtime': 1117.4422, 'train_samples_per_second': 57.202, 'train_steps_per_second': 3.58, 'train_loss': 0.1896103126069774, 'epoch': 23.0})

In [10]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary(device=0))

True
NVIDIA GeForce RTX 4070 SUPER
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3825 MiB |   3850 MiB |   5968 MiB |   2142 MiB |
|       from large pool |   3821 MiB |   3847 MiB |   5892 MiB |   2070 MiB |
|       from small pool |      3 MiB |      4 MiB |     76 MiB |     72 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   3825 MiB |   3850 MiB |   5968 MiB |   2142 MiB |
|       from large pool |   3

In [6]:
print(torch.cuda.is_available())

True


In [ ]:
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [17]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Zugriff verweigert: 'C:\\Users\\Kochana\\projects\\genres\\ast_venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu126
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu126/torchvision-0.22.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu126/torchaudio-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu126/torch-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 6.3/6.3 MB 6.6 MB/s eta 0:00:00
   ---------------------------------------- 2.8/2.8 GB 994.8 kB/s eta 0:00:00
   ---------------------------------------- 4.2/4.2 MB 7.1 MB/s eta 0:00:00
  

In [1]:
import torch
print(torch.__version__)

2.7.0+cpu


In [25]:
pip install accelerate==0.28.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
